In [ ]:
!pip install -q -U langgraph langchain-core openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 44.2 MB/s eta 0:00:00


In [ ]:
# Import os module
import os

# Used for type definition of our graph state
from typing import TypedDict

# Import LangGraph components
from langgraph.graph import StateGraph, START, END

# Import OpenAI client
# OpenRouter provides an OpenAI-compatible API
from openai import OpenAI

# Import Google Colab userdata
# This allows us to safely read the API key from Colab Secrets
from google.colab import userdata

In [ ]:
# Get OpenRouter API key from Google Colab Secrets
OPENROUTER_API_KEY = userdata.get("GenAi_Chatbot")

# Check whether API key was found
if not OPENROUTER_API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY not found. "
        "Please add it to Google Colab Secrets."
    )

print("OpenRouter API key loaded successfully.")

OpenRouter API key loaded successfully.


In [ ]:
# Create OpenRouter client
# OpenRouter provides an OpenAI-compatible API

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

print("OpenRouter client created successfully.")

OpenRouter client created successfully.


In [ ]:
# Select the model that will generate the response

MODEL_NAME = "openai/gpt-oss-20b"

print("Model selected:", MODEL_NAME)

Model selected: openai/gpt-oss-20b


In [ ]:
# ============================================================
# 1. DEFINE THE GRAPH STATE SCHEMA
# ============================================================

class AgentState(TypedDict):

    # Stores the question given by the user
    user_query: str

    # Stores the response generated by the LLM
    response: str

    # Stores whether the response passed validation
    is_valid: bool

In [ ]:
# ============================================================
# 2. DEFINE NODE FUNCTIONS
# ============================================================

# Generator Node
# This node sends the user's query to OpenRouter
# and generates an answer using the LLM.

def generate_response_node(state: AgentState):

    # Get user's question from the graph state
    query = state["user_query"]

    # Create a prompt for the LLM
    prompt = f"""
You are a helpful AI assistant.

Answer the following user question clearly and
in a simple way.

User Question:
{query}

Give a useful and meaningful answer.
"""

    # Send the prompt to OpenRouter
    response = client.chat.completions.create(

        # Select the OpenRouter model
        model=MODEL_NAME,

        # Send messages to the LLM
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        # Lower temperature gives more consistent answers
        temperature=0.3
    )

    # Extract the generated text
    generated_answer = response.choices[0].message.content

    # Return updated state
    return {
        "response": generated_answer
    }

In [ ]:
# ============================================================
# VALIDATION NODE
# ============================================================

def validation_node(state: AgentState):

    # Get the generated response
    response = state["response"]

    # Check whether response contains more than 10 characters
    valid = len(response) > 10

    # Return validation result
    return {
        "is_valid": valid
    }

In [ ]:
# ============================================================
# 3. DEFINE ROUTING LOGIC
# ============================================================

def router(state: AgentState):

    # Check validation result
    if state["is_valid"]:

        # Response is valid
        return "approved"

    else:

        # Response is not valid
        return "rejected"

In [ ]:
# ============================================================
# 4. BUILD THE LANGGRAPH STATE MACHINE
# ============================================================

# Create StateGraph using our AgentState
builder = StateGraph(AgentState)

In [ ]:
# Add Generator Node
builder.add_node(
    "generator",
    generate_response_node
)

# Add Validator Node
builder.add_node(
    "validator",
    validation_node
)

In [ ]:
# Start the graph with the generator node

builder.add_edge(
    START,
    "generator"
)

In [ ]:
# After generating the response,
# send it to the validation node

builder.add_edge(
    "generator",
    "validator"
)

In [ ]:
# ============================================================
# CONDITIONAL ROUTING
# ============================================================

builder.add_conditional_edges(

    # Routing starts from validator node
    "validator",

    # Function that decides where to go
    router,

    # Possible routes
    {
        # If approved → finish the graph
        "approved": END,

        # If rejected → go back to generator
        "rejected": "generator"
    }
)

In [ ]:
# ============================================================
# COMPILE LANGGRAPH APPLICATION
# ============================================================

app = builder.compile()

print("LangGraph compiled successfully!")

LangGraph compiled successfully!


In [ ]:
# ============================================================
# 5. EXECUTE GRAPH
# ============================================================

# Ask the user for a question

user_question = input(
    "Enter your question: "
)


# Execute LangGraph

output = app.invoke({

    "user_query": user_question,

    "response": "",

    "is_valid": False
})


# Display result

print(
    "\n--- LANGGRAPH EXECUTION COMPLETE ---"
)

print(
    "\nFinal State:"
)

print(output)


print(
    "\nAI Response:"
)

print(
    output["response"]
)

Enter your question: Tell me what is langgraph and how it works

--- LANGGRAPH EXECUTION COMPLETE ---

Final State:
{'user_query': 'Tell me what is langgraph and how it works', 'response': '**What is LangGraph?**  \nLangGraph is an open‑source framework (from the LangChain team) that lets you build language‑model applications as *graphs*.  \nInstead of writing a long linear script, you create a network of small “nodes” (functions, prompts, LLM calls, API calls, etc.) and connect them with edges that decide the next step.  \n\n**Why use it?**  \n* Keeps complex logic tidy.  \n* Lets you reuse nodes in many workflows.  \n* Gives you a visual view of the flow (via LangSmith).  \n* Handles state, memory, and branching automatically.\n\n---\n\n### How it works – a quick walk‑through\n\n| Step | What you do | What happens |\n|------|-------------|--------------|\n| **1. Define nodes** | Write small functions or LLM calls. Each node receives the current *state* and returns a new state. | Node